# Lab 06 — Erasure Coding in Practice

Erasure Coding (EC) is the technology that lets distributed storage systems achieve
high durability with far less overhead than traditional replication.

### The core idea: Reed-Solomon codes

**RS(k, m)** splits an object into **k data shards** and computes **m parity shards**
using Galois Field arithmetic. Any **k** shards (out of k+m total) are sufficient
to reconstruct the original data — even if up to **m** shards are lost.

```
Original data  ──RS(4,2)──▶  [D1][D2][D3][D4]  ←  4 data shards
                              [P1][P2]           ←  2 parity shards
                               ↓   ↓   ↓   ↓
                            drive0 1  2  3      ←  distributed across drives

If drive1 & drive2 fail: [D1][xx][xx][D4][P1][P2] → still 4 shards → reconstruct D2,D3
```

In this lab you'll upload objects of various sizes, inspect their metadata,
compare storage efficiency across EC schemes, and observe the cluster health.


---
## 0 · Prerequisites

1. RustFS is running: `docker compose up -d`
2. Virtual environment active with `boto3` installed: `uv sync`
3. Docker CLI available in PATH (for container status checks)


---
## Setup — Connect and Create Bucket


In [6]:
import boto3
import subprocess
import math
from botocore.config import Config

s3 = boto3.client(
    service_name='s3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1',
    use_ssl=False,
    config=Config(
        signature_version='s3v4',
        s3={'addressing_style': 'path'},
    ),
)

BUCKET = 'lab6-ec'

existing = {b['Name'] for b in s3.list_buckets()['Buckets']}
if BUCKET not in existing:
    s3.create_bucket(Bucket=BUCKET)
    print(f'✅ Created bucket: {BUCKET}')
else:
    print(f'✅ Bucket already exists: {BUCKET}')


✅ Created bucket: lab6-ec


---
## 1 · Verify Cluster Health

Before uploading, confirm RustFS is healthy and all drives are online.
The liveness endpoint returns HTTP 200 when the server is ready.


In [7]:
# Liveness check
result = subprocess.run(
    ['curl', '-sf', '-o', '/dev/null', '-w', '%{http_code}',
     'http://localhost:9000/minio/health/live'],
    capture_output=True, text=True,
)
code = result.stdout.strip()
print(f'RustFS liveness: HTTP {code} — {"healthy" if code == "200" else "not ready"}')

# Container status
result = subprocess.run(
    ['docker', 'compose', 'ps'],
    capture_output=True, text=True, cwd='..'
)
print('\nContainer status:')
print(result.stdout or '(no output — run from repo root)')


RustFS liveness: HTTP 403 — not ready

Container status:
NAME            IMAGE                        COMMAND                  SERVICE         CREATED          STATUS                    PORTS
rustfs-server   rustfs/rustfs:1.0.0-beta.8   "/entrypoint.sh /dat…"   rustfs-server   55 seconds ago   Up 53 seconds (healthy)   0.0.0.0:9000-9001->9000-9001/tcp, [::]:9000-9001->9000-9001/tcp



---
## 2 · Upload Objects of Varying Sizes

We upload objects ranging from 1 KB to 10 MB to observe how RustFS handles
different sizes. Internally, shard size is dynamically adjusted (64 KB – 4 MB)
based on object size.


In [8]:
# Object sizes to test: key → size in bytes
TEST_OBJECTS = {
    'ec/object_1KB.bin':   1 * 1024,
    'ec/object_10KB.bin':  10 * 1024,
    'ec/object_100KB.bin': 100 * 1024,
    'ec/object_1MB.bin':   1 * 1024 * 1024,
    'ec/object_10MB.bin':  10 * 1024 * 1024,
}

for key, size in TEST_OBJECTS.items():
    content = b'a' * size
    s3.put_object(Bucket=BUCKET, Key=key, Body=content)
    print(f'  Uploaded {key:30s}  ({size:>12,} bytes)')

print(f'\n✅ {len(TEST_OBJECTS)} objects uploaded.')


  Uploaded ec/object_1KB.bin               (       1,024 bytes)
  Uploaded ec/object_10KB.bin              (      10,240 bytes)
  Uploaded ec/object_100KB.bin             (     102,400 bytes)
  Uploaded ec/object_1MB.bin               (   1,048,576 bytes)
  Uploaded ec/object_10MB.bin              (  10,485,760 bytes)

✅ 5 objects uploaded.


---
## 3 · Inspect Object Metadata

`head_object` returns the **ETag** — a content fingerprint computed by S3.
For multipart uploads the ETag includes the number of parts; for single-part
uploads it is the MD5 of the content.

Note that `ContentLength` reflects the **logical object size** as the client sees it —
not the raw storage used (which is higher due to EC parity shards).


In [9]:
print(f'  {"Key":30s}  {"Logical size":>14}  ETag')
print('  ' + '-' * 80)
for key in TEST_OBJECTS:
    meta = s3.head_object(Bucket=BUCKET, Key=key)
    size = meta['ContentLength']
    etag = meta['ETag']
    print(f'  {key:30s}  {size:>14,} bytes  {etag}')


  Key                               Logical size  ETag
  --------------------------------------------------------------------------------
  ec/object_1KB.bin                        1,024 bytes  "c9a34cfc85d982698c6ac89f76071abd"
  ec/object_10KB.bin                      10,240 bytes  "416671d9da6b155c340c93ca08845194"
  ec/object_100KB.bin                    102,400 bytes  "302d3a0c8e319eaa95b059b346de1d1d"
  ec/object_1MB.bin                    1,048,576 bytes  "7202826a7791073fe2787f0c94603278"
  ec/object_10MB.bin                  10,485,760 bytes  "e56e104794a18df5f41f6d2d87b4cc67"


---
## 4 · Storage Efficiency Comparison

**Storage efficiency** = data shards / total shards = `k / (k + m)`.

Higher efficiency means less overhead — but fewer drives can fail before data loss.
The right trade-off depends on your cluster size and durability requirements.


In [10]:
# EC scheme comparison
# (name, data_shards k, parity_shards m, notes)
schemes = [
    ('3x Replication',      1,  2, 'Simple; any 1 of 3 copies survives'),
    ('RS(4,2)  — EC:2',     4,  2, 'RustFS default; 67% efficiency'),
    ('RS(4,4)  — EC:4',     4,  4, 'High fault tolerance; 50% efficiency'),
    ('RS(6,2)',              6,  2, 'High efficiency; 75%'),
    ('RS(10,4)',            10,  4, 'Large clusters; 71%'),
]

header = f'{"Scheme":<22}  {"k":>2}  {"m":>2}  {"n":>2}  {"Efficiency":>11}  {"Max failures":>12}'
print(header)
print('-' * len(header))
for name, k, m, note in schemes:
    n = k + m
    eff = k / n
    marker = '  ← this lab' if k == 4 and m == 2 else ''
    print(f'{name:<22}  {k:>2}  {m:>2}  {n:>2}  {eff:>10.1%}  {m:>12}{marker}')

print()
print('Interpretation: RS(4,2) stores a 10 GB dataset in 15 GB (10/0.67).')
print('3x replication stores the same dataset in 30 GB — 2× less efficient.')


Scheme                   k   m   n   Efficiency  Max failures
-------------------------------------------------------------
3x Replication           1   2   3       33.3%             2
RS(4,2)  — EC:2          4   2   6       66.7%             2  ← this lab
RS(4,4)  — EC:4          4   4   8       50.0%             4
RS(6,2)                  6   2   8       75.0%             2
RS(10,4)                10   4  14       71.4%             4

Interpretation: RS(4,2) stores a 10 GB dataset in 15 GB (10/0.67).
3x replication stores the same dataset in 30 GB — 2× less efficient.


---
## 5 · Simulating Drive Failure

In a real scenario, you can simulate a drive failure by making a volume directory
inaccessible and then verifying objects remain readable:

```bash
# Make drive1 unreadable (inside the container)
docker exec rustfs-server chmod 000 /data/drive1

# All objects still accessible (EC reconstructs from 5 surviving shards)
# ...

# Restore access
docker exec rustfs-server chmod 755 /data/drive1
```

RustFS performs **read-time repair** transparently: when it detects a missing or
corrupted shard during a GET, it reconstructs the shard from surviving ones and
writes the repaired shard back to the drive — all inline, invisible to the client.

### Storage overhead comparison for a 100 TB dataset

| Scheme | Raw data | Storage needed | Overhead |
|--------|---------|---------------|----------|
| 3× Replication | 100 TB | 300 TB | +200% |
| RS(4,2) | 100 TB | 150 TB | +50% |
| RS(6,2) | 100 TB | 133 TB | +33% |
| RS(10,4) | 100 TB | 140 TB | +40% |


---
## 6 · Cleanup


In [ ]:
response = s3.list_objects_v2(Bucket=BUCKET)
for obj in response.get('Contents', []):
    s3.delete_object(Bucket=BUCKET, Key=obj['Key'])
    print(f'🗑️  Deleted {obj["Key"]}')

s3.delete_bucket(Bucket=BUCKET)
print(f'🗑️  Deleted bucket: {BUCKET}')

print('\n✅ Cleanup complete!')


---
## 📋 Summary

| Concept | Value |
|---------|-------|
| EC scheme in this lab | RS(4,2) |
| Data shards | 4 |
| Parity shards | 2 |
| Total shards | 6 (across 4 drives) |
| Max drive failures tolerated | 2 |
| Storage efficiency | 67% |

### Key takeaways

- **EC always outperforms replication** in storage efficiency for the same fault tolerance
- The trade-off is **CPU cost**: EC requires encoding on write and reconstruction on repair
- RustFS uses SIMD-accelerated (AVX2) Reed-Solomon to keep encoding overhead minimal
- `ContentLength` in `head_object` shows the **logical** size — parity overhead is hidden

### Next steps

- Read `docs/04-erasure-coding.md` for the full mathematical explanation of Reed-Solomon
- Read `docs/05-fault-tolerance.md` for healing mechanisms and failure scenarios
